In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from base_seer_custom import Base_seer
pd.options.plotting.backend = "plotly"

In [2]:
s = Base_seer('/home/fernando/seer/SEER_1975_2016_CUSTOM_TEXTDATA')

Tabela carregada.


In [3]:
s.df = s.df.fillna(0, subset='SRV_TIME_MON')

In [65]:
# adicionar histórico do paciente
query = '''
SELECT 
--    ######################################## TARGETS #######################################
    
       CASE
         WHEN SRV_TIME_MON >= 60 THEN 1 ELSE 0 END      AS CURADO
     , SEER.SRV_TIME_MON                                AS TEMPO_SOBRE
     , CASE
         WHEN VSRTSADX = 1 THEN 1
         WHEN VSRTSADX IN (0, 8) THEN 0
         ELSE NULL END                                  AS SOBRE_COM_CANCER
         
--    ################################### DADOS DO PACIENTE #######################################
    
     , YR_BRTH                                          AS ANO_NASC
     , SEER.YEAR_DX                                     AS ANO_DIAG
     , SEER.MDXRECMP                                    AS MES_DIAG
     , AGE_DX                                           AS IDADE_DIAG
     , CAST(substring(ST_CNTY, 0, 2) AS INTEGER)        AS ESTADO
     , CAST(substring(ST_CNTY, 3, 3) AS INTEGER)        AS CIDADE
     , RACE1V                                           AS ETNIA
     , SEX                                              AS SEXO
     , MAR_STAT                                         AS ESTADO_CIVIL
     , INSREC_PUB                                       AS PLANO_SAUDE
     
--     ################################### CODIGOS DO TUMOR #######################################
     
     , PRIMSITE                                         AS CID
     , SITERWHO                                         AS CID_SEER
     , ICCC3XWHO                                        AS ICCC
     , AYASITERWHO                                      AS AYA
     , HISTO3V                                          AS HIST
     , BEHO3V                                           AS HIST_COMP
     , HISTREC                                          AS HIST_SEER
     , HISTRECB                                         AS HIST_SEER_CEREBRO

--     ############################## ESPECIFICAÇÕES DO TUMOR ####################################### 
     , COALESCE(
        SCSSM2KO
      , DSS1977S
      , SSS77VZ
      , SSSM2KPZ
      , SUMM2K
      , CASE
          WHEN HST_STGA = 4 THEN 7 ELSE HST_STGA END)   AS ESTAGIO_SUMARIZADO

FROM SEER
LIMIT 10
'''
s.spark.sql(query).show()

+------+-----------+----------------+--------+--------+--------+----------+------+------+-----+----+------------+-----------+----+--------+----+----+----+---------+---------+-----------------+------------------+
|CURADO|TEMPO_SOBRE|SOBRE_COM_CANCER|ANO_NASC|ANO_DIAG|MES_DIAG|IDADE_DIAG|ESTADO|CIDADE|ETNIA|SEXO|ESTADO_CIVIL|PLANO_SAUDE| CID|CID_SEER|ICCC| AYA|HIST|HIST_COMP|HIST_SEER|HIST_SEER_CEREBRO|ESTAGIO_SUMARIZADO|
+------+-----------+----------------+--------+--------+--------+----------+------+------+-----+----+------------+-----------+----+--------+----+----+----+---------+---------+-----------------+------------------+
|     0|          8|            null|    1918|    2001|      10|        83|    34|     7|    1|   2|           1|       null|C184|   21045|  98|  42|8140|        3|        5|               98|                 1|
|     0|          8|            null|    1914|    2001|       1|        86|    34|    21|    1|   2|           5|       null|C184|   21045|  98|  42|848

In [64]:
query = '''
SELECT DAJCCT
     , DAJCC7T
, ADJTM_6VALUE
T_VALUE
DASRCT


FROM SEER
limit 100
'''
s.spark.sql(query).show(100)

+------------------+--------+--------+-------+--------+------+-------------------------------------------------+
|ESTAGIO_SUMARIZADO|SCSSM2KO|DSS1977S|SSS77VZ|SSSM2KPZ|SUMM2K|CASE WHEN (HST_STGA = 4) THEN 7 ELSE HST_STGA END|
+------------------+--------+--------+-------+--------+------+-------------------------------------------------+
|                 1|    null|    null|   null|       1|     1|                                                1|
|                 1|    null|    null|   null|       1|     1|                                                1|
|                 0|    null|    null|   null|       0|     0|                                                0|
|                 1|       1|       1|   null|    null|     1|                                                1|
|                 7|       7|       7|   null|    null|     7|                                                7|
|                 2|       2|    null|   null|    null|     2|                                  

In [51]:
query = '''
SELECT SUMM2K
     , count(*)
FROM SEER
group by SUMM2K
'''
s.spark.sql(query).show(50)

+------+--------+
|SUMM2K|count(1)|
+------+--------+
|  null| 3829478|
|     1| 3146620|
|     8|       1|
|     7| 1345120|
|     2| 1389898|
|     0|  739592|
+------+--------+

